# SEC CFR Title 17 + FIBO → Neo4j Knowledge Graph

Builds a Neo4j graph combining the FIBO ontology with SEC CFR Title 17 (eCFR).

**Sections:**
1. Download Data — FIBO archive + eCFR Title 17 JSON/XML
2. Load FIBO into Neo4j — parse RDF → Python dict → clean property graph via rdflib-neo4j (IGNORE strategy)
3. Process and Load SEC Data — extract text from XML, convert JSON → RDF TTL, load via rdflib-neo4j (SHORTEN strategy)
4. Infer GOVERNS Relationships — match FIBO concept labels against CFR section text

Run the **Configuration** cell first, then execute sections in order.

In [1]:
from pathlib import Path

# === Configuration ===
SEC_DATA_DIR = Path('/Users/michaelmoore/GitHub/neo4j-fibo-sec/data/sec')
SEC_DATA_DIR.mkdir(parents=True, exist_ok=True)
FIBO_DATA_DIR = Path('/Users/michaelmoore/GitHub/neo4j-fibo-sec/data/fibo')
FIBO_DATA_DIR.mkdir(parents=True, exist_ok=True)

# FIBO (GitHub archive)
FIBO_GH_ZIP_URL = 'https://github.com/edmcouncil/fibo/archive/refs/heads/master.zip'
USE_GITHUB_FIBO = True
FIBO_ZIP_URL = ''  # Set this to a direct URL if USE_GITHUB_FIBO = False

# eCFR
ECFR_TITLE = '17'
ECFR_DATE = '2026-02-25'  # adjust if unavailable

# Identify your requests
USER_AGENT = 'Michael Moore (michael.moore@neo4j.com)'

# Neo4j connection
NEO4J_URI = 'neo4j://127.0.0.1:7687'
NEO4J_USER = 'neo4j'
NEO4J_PWD = 'testtest'
NEO4J_DB = 'neo4j'

print('SEC data dir:', SEC_DATA_DIR)
print('FIBO data dir:', FIBO_DATA_DIR)


SEC data dir: /Users/michaelmoore/GitHub/neo4j-fibo-sec/data/sec
FIBO data dir: /Users/michaelmoore/GitHub/neo4j-fibo-sec/data/fibo


In [2]:
#genai credentials
import pandas as pd
from openai import OpenAI
from neo4j import GraphDatabase
import time
from typing import List, Dict, Any
import json
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# OpenAI Configuration (using OpenAI directly, not Azure)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# For Neo4j vector search, we still need Azure OpenAI credentials
# (since Neo4j is configured to use Azure OpenAI for embeddings)
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = "https://openai-ml-emerson.openai.azure.com/"
AZURE_API_VERSION = "2024-02-01"
EMBEDDING_DEPLOYMENT_NAME = "text-embedding-3-large"
EMBEDDING_DIMENSIONS = 3072

# OpenAI Chat model configuration
CHAT_MODEL_NAME = "gpt-4o-mini"  # OpenAI model name
CHAT_MODEL_TEMPERATURE = 0.0     # 0.0-0.2 recommended for fuzzy NER (handles misspellings)
CHAT_MODEL_MAX_TOKENS = 1000

#temperature = 0.0
#top_p = 1.0
#frequency_penalty = 0.0
#presence_penalty = 0.0
#seed = 42

# Initialize OpenAI client (direct OpenAI, not Azure)
client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ Connected to OpenAI and Neo4j")
print(f"📝 Chat model: {CHAT_MODEL_NAME}")
print(f"🌡️  Temperature: {CHAT_MODEL_TEMPERATURE} (allows fuzzy matching)")
print(f"🔢 Embedding (via Azure): {EMBEDDING_DEPLOYMENT_NAME}")

✅ Connected to OpenAI and Neo4j
📝 Chat model: gpt-4o-mini
🌡️  Temperature: 0.0 (allows fuzzy matching)
🔢 Embedding (via Azure): text-embedding-3-large


In [39]:
# If needed, install dependencies
# %pip install requests beautifulsoup4 lxml pandas rdflib tqdm

In [40]:
import io
import json
import zipfile
import requests
from tqdm import tqdm

def download_file(url, dest_path, chunk_size=1024*1024):
    dest_path = Path(dest_path)
    print(f'Downloading file: {url} -> {dest_path}')
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, headers={'User-Agent': 'sec-fibo-loader/1.0', 'Accept': '*/*'}) as r:
        r.raise_for_status()
        total = int(r.headers.get('Content-Length', 0))
        with open(dest_path, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as pbar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    return dest_path

def unzip(zip_path, dest_dir):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(dest_dir)
    return dest_dir

## 1) Download Data

Download the FIBO ontology archive from EDM Council GitHub and CFR Title 17
from the eCFR API (structure JSON + full XML bulk download).

In [28]:
fibo_dir = FIBO_DATA_DIR
fibo_dir.mkdir(parents=True, exist_ok=True)

if USE_GITHUB_FIBO:
    fibo_zip = FIBO_DATA_DIR / 'fibo_github.zip'
    if not fibo_zip.exists():
        download_file(FIBO_GH_ZIP_URL, fibo_zip)
    unzip(fibo_zip, fibo_dir)
    print('Extracted GitHub FIBO to', fibo_dir)
else:
    if not FIBO_ZIP_URL:
        raise ValueError('Set FIBO_ZIP_URL to the EDM Council product download URL')
    fibo_zip = FIBO_DATA_DIR / 'fibo_product.zip'
    if not fibo_zip.exists():
        download_file(FIBO_ZIP_URL, fibo_zip)
    unzip(fibo_zip, fibo_dir)
    print('Extracted EDM Council FIBO to', fibo_dir)

print('FIBO root:', fibo_dir)


Extracted GitHub FIBO to /Users/michaelmoore/GitHub/neo4j-fibo-sec/data/fibo
FIBO root: /Users/michaelmoore/GitHub/neo4j-fibo-sec/data/fibo


In [29]:
# Ensure config cell has been run; provide safe defaults if not
if 'SEC_DATA_DIR' not in globals():
    from pathlib import Path
    SEC_DATA_DIR = Path.cwd().parent / 'data' / 'sec'
    SEC_DATA_DIR.mkdir(parents=True, exist_ok=True)
if 'ECFR_TITLE' not in globals():
    ECFR_TITLE = '17'
if 'ECFR_DATE' not in globals():
    ECFR_DATE = '2026-02-25'
if 'USER_AGENT' not in globals():
    USER_AGENT = 'sec-fibo-loader/1.0 (contact: you@example.com)'

import json
import requests
import time
import random
import re


HEADERS = {
    'User-Agent': USER_AGENT,
    'Accept': 'application/json'
}

def ecfr_get(path, params=None, max_retries=5, base_delay=1.0):
    url = 'https://www.ecfr.gov' + path
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=30)
            if r.status_code in (429, 500, 502, 503, 504):
                raise RuntimeError(f'HTTP {r.status_code}')
            r.raise_for_status()
            return r
        except Exception:
            if attempt == max_retries - 1:
                raise
            sleep_for = base_delay * (2 ** attempt) + random.uniform(0, 0.5)
            time.sleep(sleep_for)

def extract_dates(obj):
    dates = set()
    if isinstance(obj, dict):
        for v in obj.values():
            dates.update(extract_dates(v))
    elif isinstance(obj, list):
        for v in obj:
            dates.update(extract_dates(v))
    elif isinstance(obj, str):
        for m in re.findall(r'\d{4}-\d{2}-\d{2}', obj):
            dates.add(m)
    return dates

# Connectivity test: list available titles
print('Fetching titles list...')
titles_resp = ecfr_get('/api/versioner/v1/titles.json')
titles_json = titles_resp.json()
print('Titles status:', titles_resp.status_code)

# Try to infer a valid date if ECFR_DATE fails
candidate_dates = sorted(extract_dates(titles_json))
if candidate_dates:
    print('Example dates from titles.json:', candidate_dates[-3:])

def try_structure(date_str):
    paths = [
        f'/api/versioner/v1/structure/{date_str}/title-{ECFR_TITLE}.json',
        f'/api/versioner/v1/structure/{date_str}/title/{ECFR_TITLE}.json',
        f'/api/versioner/v1/structure/{date_str}/title-{ECFR_TITLE}',
        f'/api/versioner/v1/structure/{date_str}/title/{ECFR_TITLE}',
    ]
    last_err = None
    for p in paths:
        try:
            r = ecfr_get(p)
            return p, r
        except Exception as e:
            last_err = e
            continue
    raise last_err

# Fetch structure for Title 17 at a point-in-time date
print('Fetching Title 17 structure...')
try:
    used_path, structure_resp = try_structure(ECFR_DATE)
    used_date = ECFR_DATE
except Exception:
    if candidate_dates:
        fallback_date = candidate_dates[-1]
        print('ECFR_DATE failed; trying fallback date:', fallback_date)
        used_path, structure_resp = try_structure(fallback_date)
        used_date = fallback_date
    else:
        raise

print('Structure path used:', used_path)
structure = structure_resp.json()

out_json = SEC_DATA_DIR / f'ecfr_title{ECFR_TITLE}_structure_{used_date}.json'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(structure, f, indent=2)
print('Saved', out_json)


Fetching titles list...
Titles status: 200
Example dates from titles.json: ['2026-02-23', '2026-02-24', '2026-02-25']
Fetching Title 17 structure...
Structure path used: /api/versioner/v1/structure/2026-02-25/title-17.json
Saved /Users/michaelmoore/GitHub/neo4j-fibo-sec/data/sec/ecfr_title17_structure_2026-02-25.json


### Optional: Download Full Title 17 XML (Bulk)

This pulls the entire Title 17 XML for the selected date in one request.

In [30]:
# Ensure config cell has been run; provide safe defaults if not
if 'SEC_DATA_DIR' not in globals():
    from pathlib import Path
    SEC_DATA_DIR = Path('/Users/michaelmoore/GitHub/neo4j-fibo-sec/data/sec')
    SEC_DATA_DIR.mkdir(parents=True, exist_ok=True)
if 'ECFR_TITLE' not in globals():
    ECFR_TITLE = '17'
if 'ECFR_DATE' not in globals():
    ECFR_DATE = '2026-02-25'
if 'USER_AGENT' not in globals():
    USER_AGENT = 'sec-fibo-loader/1.0 (contact: you@example.com)'

import requests
from tqdm import tqdm
import re

# If a structure JSON exists, prefer its date to avoid 404s
struct_glob = list(SEC_DATA_DIR.glob(f'ecfr_title{ECFR_TITLE}_structure_*.json'))
if struct_glob:
    # pick the most recent by filename
    struct_glob.sort()
    m = re.search(r'structure_(\d{4}-\d{2}-\d{2})\.json$', struct_glob[-1].name)
    if m:
        ECFR_DATE = m.group(1)

full_url = f'https://www.ecfr.gov/api/versioner/v1/full/{ECFR_DATE}/title-{ECFR_TITLE}.xml'
out_xml = SEC_DATA_DIR / f'ecfr_title{ECFR_TITLE}_full_{ECFR_DATE}.xml'

print(f'Downloading file: {full_url} -> {out_xml}')
with requests.get(full_url, stream=True, headers={'User-Agent': USER_AGENT, 'Accept': 'application/xml'}, timeout=60) as r:
    r.raise_for_status()
    total = int(r.headers.get('Content-Length', 0))
    with open(out_xml, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as pbar:
        for chunk in r.iter_content(chunk_size=1024*1024):
            if chunk:
                f.write(chunk)
                pbar.update(len(chunk))

print('Saved', out_xml)


100%|██████████████████████████████████████| 17.2M/17.2M [00:03<00:00, 5.38MB/s]

Saved /Users/michaelmoore/GitHub/neo4j-fibo-sec/data/sec/ecfr_title17_full_2026-02-25.xml


In [31]:
# Integrity check for structure JSON and full XML
from pathlib import Path

struct_path = SEC_DATA_DIR / f'ecfr_title{ECFR_TITLE}_structure_{ECFR_DATE}.json'
xml_path = SEC_DATA_DIR / f'ecfr_title{ECFR_TITLE}_full_{ECFR_DATE}.xml'

def report(p):
    if p.exists():
        size_mb = p.stat().st_size / (1024*1024)
        print(f'OK: {p} ({size_mb:.2f} MB)')
    else:
        print(f'MISSING: {p}')

report(struct_path)
report(xml_path)


OK: /Users/michaelmoore/GitHub/neo4j-fibo-sec/data/sec/ecfr_title17_structure_2026-02-25.json (2.21 MB)
OK: /Users/michaelmoore/GitHub/neo4j-fibo-sec/data/sec/ecfr_title17_full_2026-02-25.xml (16.45 MB)


## 2) Load FIBO into Neo4j

Parse all FIBO `.rdf` / `.ttl` files into a unified rdflib Graph, extract
`skos:Concept`, `skos:ConceptScheme`, and `owl:Class` nodes with their labels
and definitions into Python dicts, then build a clean property-graph subgraph
and load it into Neo4j via rdflib-neo4j using `HANDLE_VOCAB_URI_STRATEGY.IGNORE`.

Result: nodes labelled `Concept`, `ConceptScheme`, `Class` with rel types
`BROADER_THAN`, `SUBCLASS_OF`, `IN_SCHEME`, etc.

In [66]:
# Quick connection test — verify Neo4j is reachable before loading data
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PWD))
with driver.session(database=NEO4J_DB) as session:
    result = session.run('MATCH (n) RETURN count(n) AS nodes')
    print('Nodes:', result.single()['nodes'])
    result = session.run('MATCH ()-[r]->() RETURN count(r) AS rels')
    print('Relationships:', result.single()['rels'])
driver.close()


Nodes: 0
Relationships: 0


In [67]:
import warnings
import logging
from pathlib import Path
from rdflib import Graph, URIRef, Literal, RDF, RDFS, OWL
from rdflib.namespace import SKOS
from tqdm import tqdm

logging.getLogger('rdflib').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', message='.*lexical form.*')

FIBO_ROOT = Path('/Users/michaelmoore/GitHub/neo4j-fibo-sec/data/fibo/fibo-master')
fibo_files = sorted(FIBO_ROOT.rglob('*.rdf')) + sorted(FIBO_ROOT.rglob('*.ttl'))
print(f'Files found: {len(fibo_files)}')

g = Graph()
errors = []
for f in tqdm(fibo_files, desc='Parsing FIBO', unit='file'):
    try:
        g.parse(str(f))
    except Exception as e:
        errors.append(f.name)

print(f'Total triples : {len(g):,}')
print(f'Parse errors  : {len(errors)}')


Files found: 296


Parsing FIBO: 100%|████████████████████████| 296/296 [00:02<00:00, 132.68file/s]

Total triples : 129,863
Parse errors  : 0


In [68]:
from collections import defaultdict

def local_name(uri):
    s = str(uri)
    return s.split('#')[-1] if '#' in s else s.rstrip('/').split('/')[-1]

def get_str(g, subj, *preds):
    for pred in preds:
        for obj in g.objects(subj, pred):
            if isinstance(obj, Literal):
                return str(obj)
    return None

def get_all_str(g, subj, pred):
    return [str(o) for o in g.objects(subj, pred) if isinstance(o, Literal)]

# FIBO entity types → Neo4j label
# owl:Class           → Class           (ontology classes / concept definitions)
# owl:NamedIndividual → Individual       (specific instances: CFTC, NYSE, USD, …)
# skos:ConceptScheme  → ConceptScheme   (top-level scheme nodes)
# skos:Concept        → Class           (rare in FIBO, treated as class-level entity)
NODE_TYPES = {
    SKOS.Concept:        'Class',
    SKOS.ConceptScheme:  'ConceptScheme',
    OWL.Class:           'Class',
    OWL.NamedIndividual: 'Individual',
}

nodes = {}  # uri_str -> dict
for rdf_type, neo4j_label in NODE_TYPES.items():
    for subj in g.subjects(RDF.type, rdf_type):
        if not isinstance(subj, URIRef):
            continue
        uri_str = str(subj)
        if uri_str in nodes:
            continue  # deduplicate — first matching type wins
        pref = get_str(g, subj, SKOS.prefLabel, RDFS.label)
        if not pref:
            continue  # skip unlabelled resources
        nodes[uri_str] = {
            'uri':        uri_str,
            'localName':  local_name(subj),
            'prefLabel':  pref,
            'nodeLabel':  neo4j_label,
            'definition': get_str(g, subj, SKOS.definition, RDFS.comment),
            'altLabels':  get_all_str(g, subj, SKOS.altLabel),
            'notation':   get_str(g, subj, SKOS.notation),
        }

by_label = defaultdict(int)
for n in nodes.values():
    by_label[n['nodeLabel']] += 1

print(f'Nodes to create: {sum(by_label.values()):,}')
for label, count in sorted(by_label.items(), key=lambda x: -x[1]):
    print(f'  :{label:<20} {count:>6,}')


Nodes to create: 14,570
  :Individual           11,627
  :Class                 2,941
  :ConceptScheme             2


In [69]:
import re
from collections import defaultdict, Counter

def to_rel_type(uri_str):
    """Derive UPPER_SNAKE_CASE Neo4j rel type from a predicate URI."""
    local = uri_str.split('#')[-1] if '#' in uri_str else uri_str.split('/')[-1]
    # camelCase → UPPER_SNAKE_CASE
    s = re.sub(r'([a-z])([A-Z])', r'\1_\2', local)
    return s.upper()

# Canonical overrides for standard OWL/RDFS predicates
CANONICAL_REL = {
    str(RDFS.subClassOf):      'SUBCLASS_OF',
    str(OWL.equivalentClass):  'EQUIVALENT_TO',
    str(OWL.disjointWith):     'DISJOINT_WITH',
    str(OWL.inverseOf):        'INVERSE_OF',
    str(RDFS.seeAlso):         'SEE_ALSO',
}

# ── Discover all predicates connecting two extracted nodes ────────────────────
# rdf:type is excluded — already handled as node labels.
RDF_TYPE = str(RDF.type)

pred_uris = set()
for s, p, o in g:
    if not isinstance(s, URIRef) or not isinstance(o, URIRef):
        continue
    if str(p) == RDF_TYPE:
        continue
    if str(s) in nodes and str(o) in nodes:
        pred_uris.add(str(p))

# Build uri → rel_type mapping
FIBO_RELS = {uri: CANONICAL_REL.get(uri, to_rel_type(uri)) for uri in pred_uris}

# ── Extract all relationships ─────────────────────────────────────────────────
rels_by_type = defaultdict(list)

for pred_uri, rel_type in FIBO_RELS.items():
    for s, o in g.subject_objects(URIRef(pred_uri)):
        if not isinstance(s, URIRef) or not isinstance(o, URIRef):
            continue
        if str(s) not in nodes or str(o) not in nodes:
            continue
        rels_by_type[rel_type].append({'from': str(s), 'to': str(o)})

total_rels = sum(len(v) for v in rels_by_type.values())
print(f'Relationships to create : {total_rels:,}  across {len(rels_by_type)} types')
print()
for rel_type, pairs in sorted(rels_by_type.items(), key=lambda x: -len(x[1])):
    print(f'  {rel_type:<35} {len(pairs):>6,}')


Relationships to create : 23,945  across 108 types

  SUBCLASS_OF                          3,717
  IDENTIFIES                           3,405
  IS_MEMBER_OF                         3,298
  IS_MANAGED_BY                        2,840
  OPERATES_IN_MUNICIPALITY             2,811
  HAS_MARKET_IDENTIFIER_CODE_STATUS    2,809
  IS_PART_OF                           1,275
  IS_PLAYED_BY                           735
  HAS_REFERENCE_CURRENCY                 529
  HAS_JURISDICTION                       238
  IS_JURISDICTION_OF                     209
  IS_REPRESENTED_BY                      209
  DENOTES                                185
  DISJOINT_WITH                          175
  IS_DEFINED_IN                          128
  HAS_TENOR                              116
  IS_PROVIDED_BY                         102
  IS_NARROWER_THAN                        64
  HAS_MEMBER                              63
  HAS_HEADQUARTERS_ADDRESS                63
  IS_REGISTERED_IN                        62
  H

In [70]:
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import RDF, RDFS, OWL, SKOS, XSD
from rdflib_neo4j import Neo4jStoreConfig, Neo4jStore, HANDLE_VOCAB_URI_STRATEGY
from tqdm import tqdm

# ── Custom namespace — local names become Neo4j labels / rel-types / props ──
# With HANDLE_VOCAB_URI_STRATEGY.IGNORE every URI reduces to its local name:
#   FIBO_PG.Class        → label  Class
#   FIBO_PG.Individual   → label  Individual
#   FIBO_PG.SUBCLASS_OF  → rel    SUBCLASS_OF
#   FIBO_PG.IDENTIFIES   → rel    IDENTIFIES
#   FIBO_PG.prefLabel    → prop   prefLabel
FIBO_PG = Namespace('https://fibo.neo4j/pg#')

# ── Node label → rdf:type URI ─────────────────────────────────────────────────
PG_TYPES = {
    'Class':         FIBO_PG.Class,
    'Individual':    FIBO_PG.Individual,
    'ConceptScheme': FIBO_PG.ConceptScheme,
}

# ── Predicate URIs built dynamically from discovered rels_by_type ────────────
# Each rel_type (e.g. 'SUBCLASS_OF') maps to FIBO_PG.SUBCLASS_OF so that
# IGNORE strategy yields the clean Neo4j rel type string.
PG_PREDS = {rel_type: FIBO_PG[rel_type] for rel_type in rels_by_type}

# ── Build intermediate rdflib Graph ──────────────────────────────────────────
pg = Graph()
for n in nodes.values():
    subj = URIRef(n['uri'])
    pg.add((subj, RDF.type,          PG_TYPES[n['nodeLabel']]))
    pg.add((subj, FIBO_PG.prefLabel, Literal(n['prefLabel'])))
    pg.add((subj, FIBO_PG.localName,  Literal(n['localName'])))
    if n.get('definition'):
        pg.add((subj, FIBO_PG.definition, Literal(n['definition'])))
    if n.get('notation'):
        pg.add((subj, FIBO_PG.notation,   Literal(n['notation'])))
    for alt in (n.get('altLabels') or []):
        pg.add((subj, FIBO_PG.altLabel,   Literal(alt)))

for rel_type, pairs in rels_by_type.items():
    pred = PG_PREDS[rel_type]
    for pair in pairs:
        pg.add((URIRef(pair['from']), pred, URIRef(pair['to'])))

print(f'PG graph triples : {len(pg):,}')
print(f'  Nodes          : {len(nodes):,}')
print(f'  Relationships  : {sum(len(v) for v in rels_by_type.values()):,}  across {len(rels_by_type)} types')

# ── Init rdflib-neo4j store with IGNORE strategy ──────────────────────────────
if 'neo4j_store' in globals():
    try:
        neo4j_store.close(True)
    except Exception:
        pass

config = Neo4jStoreConfig(
    auth_data={'uri': NEO4J_URI, 'user': NEO4J_USER, 'pwd': NEO4J_PWD, 'database': NEO4J_DB},
    handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.IGNORE,
    batching=True,
    batch_size=5000,
)
neo4j_store = Neo4jStore(config=config)
neo4j_store.open(configuration=None, create=True)

# ── Load via rdflib-neo4j ─────────────────────────────────────────────────────
triples = list(pg)
for triple in tqdm(triples, desc='Loading FIBO PG', unit='triple'):
    neo4j_store.add(triple)
neo4j_store.commit()
neo4j_store.close(True)

print(f'Loaded {len(triples):,} triples via rdflib-neo4j (IGNORE strategy)')


PG graph triples : 72,585
  Nodes          : 14,570
  Relationships  : 23,945  across 108 types
Uniqueness constraint on :Resource(uri) found. 
                
                
The store is now: Open
Uniqueness constraint on :Resource(uri) found. 
                
                
The store is now: Open


Loading FIBO PG: 100%|█████████████| 72585/72585 [00:07<00:00, 10278.09triple/s]


The store is now: Closed
IMPORTED 72585 TRIPLES
Loaded 72,585 triples via rdflib-neo4j (IGNORE strategy)


In [71]:
# Verify FIBO property graph load
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PWD))
with driver.session(database=NEO4J_DB) as session:
    for label in ('Class', 'Individual', 'ConceptScheme'):
        c = session.run(f'MATCH (n:{label}) RETURN count(n) AS c').single()['c']
        print(f'  :{label:<20} {c:>6,}')
    print()
    for rel_type in sorted(rels_by_type):
        c = session.run(f'MATCH ()-[r:{rel_type}]->() RETURN count(r) AS c').single()['c']
        print(f'  :{rel_type:<22} {c:>6,}')
driver.close()


  :Class                 2,941
  :Individual           11,627
  :ConceptScheme             2

  :APPLIES_TO                  2
  :COMPRISES                  58
  :DENOTES                   185
  :DESCRIBES                   3
  :DIFFERENT_FROM              9
  :DISJOINT_WITH             175
  :EXCHANGES                   2
  :HAS_ACCRUAL_BASIS           2
  :HAS_BUSINESS_CENTER        23
  :HAS_BUSINESS_DAY_CONVENTION      4
  :HAS_BUYER                   2
  :HAS_CONTRACT_PARTY          2
  :HAS_CURRENCY               22
  :HAS_DATE_ADDED              5
  :HAS_DATE_ESTABLISHED       28
  :HAS_DATE_INSURED            5
  :HAS_DATE_OF_BIRTH           2
  :HAS_DATE_OF_INCORPORATION      8
  :HAS_DATE_OF_REGISTRATION     13
  :HAS_DIRECT_OWNERSHIP        3
  :HAS_DIRECT_OWNING_ENTITY      3
  :HAS_DOMESTIC_ULTIMATE_PARENT     15
  :HAS_EFFECTIVE_DATE          2
  :HAS_EMPLOYED_PARTY          2
  :HAS_EMPLOYING_PARTY         2
  :HAS_ENTITY_STATUS          17
  :HAS_GLOBAL_ULTIMATE_PARENT 

In [ ]:
# Tag all FIBO-loaded nodes with a secondary :FIBO label
# Needed for queries like MATCH (f:FIBO:Class) to distinguish FIBO concepts
# from any future Class nodes loaded from other sources.
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PWD))
with driver.session(database=NEO4J_DB) as s:
    for label in ("Class", "Individual", "ConceptScheme"):
        r = s.run(f"""
            MATCH (n:{label})
            WHERE n.uri CONTAINS 'edmcouncil.org'
               OR n.uri CONTAINS 'omg.org/spec'
            SET n:FIBO
            RETURN count(n) AS c
        """).single()["c"]
        print(f"  :{label} tagged :FIBO  {r:,}")
driver.close()


## 3) Process and Load SEC eCFR Data

Extract section text from the full Title 17 XML, then load the eCFR hierarchy
directly from the structure JSON into Neo4j using the same dictionary/IGNORE
approach as FIBO (Section 2).

Result: `Title / Chapter / Part / Subpart / Section` nodes linked by `BROADER`,
with `Text` nodes carrying raw section text linked by `HAS_TEXT`.

### 3a) Extract Section Text from Full XML

Parse the Title 17 bulk XML to extract raw text for each section identifier.

In [74]:
from lxml import etree
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, DCTERMS
from urllib.parse import quote
import re

ECFR = Namespace('https://www.ecfr.gov/cfr/')

xml_files = sorted(SEC_DATA_DIR.glob(f'ecfr_title{ECFR_TITLE}_full_*.xml'))
if not xml_files:
    raise FileNotFoundError('Full XML not found. Download it first.')
xml_path = xml_files[-1]

print('Parsing XML:', xml_path)
tree = etree.parse(str(xml_path))
root = tree.getroot()

def clean_text(text):
    if text is None:
        return ''
    return re.sub(r'\s+', ' ', text).strip()

def localname(el):
    return etree.QName(el).localname

heading_tags = {'HEAD', 'HED', 'HD1', 'HD2'}
sec_pattern = re.compile(r'§\s*([0-9A-Za-z][0-9A-Za-z\.\-\(\)]+)')

def find_container(el):
    cur = el
    while cur is not None:
        ln = localname(cur)
        if ln.startswith('DIV'):
            return cur
        cur = cur.getparent()
    return el.getparent() or el

g = Graph()
g.bind('ecfr', ECFR)
g.bind('dct', DCTERMS)

count = 0
seen = set()

for el in root.iter():
    if localname(el) not in heading_tags:
        continue
    head_text = clean_text(''.join(el.itertext()))
    m = sec_pattern.search(head_text)
    if not m:
        continue
    secno = m.group(1)
    container = find_container(el)
    cid = id(container)
    if cid in seen:
        continue
    seen.add(cid)

    text = clean_text(' '.join(container.itertext()))
    if not text:
        continue

    safe_sec = quote(secno, safe='.-_~')
    sec_uri = URIRef(ECFR + f'section/{safe_sec}')
    text_uri = URIRef(ECFR + f'section/{safe_sec}/text')
    g.add((text_uri, RDF.type, ECFR.Text))
    g.add((text_uri, ECFR.text, Literal(text)))
    g.add((sec_uri, ECFR.hasText, text_uri))
    count += 1

print('Sections with text:', count)

m = re.search(r'full_(\d{4}-\d{2}-\d{2})\.xml$', xml_path.name)
out_date = m.group(1) if m else ECFR_DATE
out_ttl = SEC_DATA_DIR / f'ecfr_title{ECFR_TITLE}_text_{out_date}.ttl'
g.serialize(out_ttl, format='turtle')
print('Saved', out_ttl)


Parsing XML: /Users/michaelmoore/GitHub/neo4j-fibo-sec/data/sec/ecfr_title17_full_2026-02-25.xml
Sections with text: 3532
Saved /Users/michaelmoore/GitHub/neo4j-fibo-sec/data/sec/ecfr_title17_text_2026-02-25.ttl


### 3b) Convert Structure JSON to RDF

Walk the eCFR hierarchy JSON (Title → Chapter → Part → Subpart → Section) and emit RDF Turtle.

In [75]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, SKOS, DCTERMS
import hashlib
import json
import re
from urllib.parse import quote

ECFR = Namespace('https://www.ecfr.gov/cfr/')

def node_uri(node, parent_uri=None):
    node_type = (node.get('type') or node.get('node_type') or 'node').lower()
    identifier = node.get('identifier') or node.get('node_id') or node.get('slug')
    if identifier:
        safe_id = quote(str(identifier), safe='.-_~')
        return URIRef(ECFR + f'{node_type}/{safe_id}')
    label = node.get('label') or node.get('heading') or node.get('title') or ''
    base = f'{node_type}|{label}|{parent_uri or ""}'
    h = hashlib.sha1(base.encode('utf-8')).hexdigest()
    return URIRef(ECFR + f'{node_type}/hash/{h}')

def node_label(node):
    return node.get('label') or node.get('heading') or node.get('title') or node.get('name') or ''

def node_notation(node):
    return node.get('number') or node.get('section') or node.get('part') or node.get('chapter') or ''

def add_volume_nodes(g, uri, volumes):
    for v in volumes or []:
        v_str = str(v).strip()
        if not v_str:
            continue
        v_uri = URIRef(ECFR + f'volume/{v_str}')
        g.add((v_uri, RDF.type, ECFR.Volume))
        g.add((v_uri, SKOS.notation, Literal(v_str)))
        g.add((uri, ECFR.inVolume, v_uri))

def add_descendant_range(g, uri, node):
    dr = node.get('descendant_range')
    if not dr:
        return
    node_type = (node.get('type') or node.get('node_type') or 'node').lower()
    identifier = node.get('identifier') or node.get('node_id') or node.get('slug') or ''
    h = hashlib.sha1(f'{node_type}|{identifier}|{dr}'.encode('utf-8')).hexdigest()
    dr_uri = URIRef(ECFR + f'descendant_range/{h}')
    g.add((dr_uri, RDF.type, ECFR.DescendantRange))
    g.add((dr_uri, SKOS.prefLabel, Literal(dr)))
    g.add((uri, ECFR.descendantRange, dr_uri))

def walk(node, g, parent_uri=None):
    uri = node_uri(node, parent_uri)
    ntype = (node.get('type') or node.get('node_type') or 'Node').title()
    g.add((uri, RDF.type, ECFR[ntype]))
    label = node_label(node)
    if label:
        g.add((uri, SKOS.prefLabel, Literal(label)))
    notation = node_notation(node)
    if notation:
        g.add((uri, SKOS.notation, Literal(str(notation))))
    if parent_uri:
        g.add((uri, SKOS.broader, parent_uri))
    add_volume_nodes(g, uri, node.get('volumes'))
    add_descendant_range(g, uri, node)
    for child in node.get('children', []) or []:
        walk(child, g, uri)

structure_files = sorted(SEC_DATA_DIR.glob(f'ecfr_title{ECFR_TITLE}_structure_*.json'))
structure_path = structure_files[-1] if structure_files else None
if not structure_path:
    raise FileNotFoundError('Structure JSON not found in SEC_DATA_DIR')

print('Loading structure JSON...', structure_path)
structure = json.loads(structure_path.read_text())
g = Graph()
g.bind('ecfr', ECFR)
g.bind('skos', SKOS)
g.bind('dct', DCTERMS)

print('Building RDF graph...')
walk(structure, g, None)

m = re.search(r'structure_(\d{4}-\d{2}-\d{2})\.json$', structure_path.name)
out_date = m.group(1) if m else ECFR_DATE
out_ttl = SEC_DATA_DIR / f'ecfr_title{ECFR_TITLE}_structure_{out_date}.ttl'
g.serialize(out_ttl, format='turtle')
print('Saved', out_ttl)


Loading structure JSON... /Users/michaelmoore/GitHub/neo4j-fibo-sec/data/sec/ecfr_title17_structure_2026-02-25.json
Building RDF graph...
Saved /Users/michaelmoore/GitHub/neo4j-fibo-sec/data/sec/ecfr_title17_structure_2026-02-25.ttl


### 3c) Pre-load Checklist

Verify all generated TTL files are present before loading.

In [76]:
# Ready-to-load checklist
from pathlib import Path

struct_glob = list(SEC_DATA_DIR.glob(f'ecfr_title{ECFR_TITLE}_structure_*.json'))
xml_glob = list(SEC_DATA_DIR.glob(f'ecfr_title{ECFR_TITLE}_full_*.xml'))
ttl_glob = list(SEC_DATA_DIR.glob(f'ecfr_title{ECFR_TITLE}_structure_*.ttl'))
text_glob = list(SEC_DATA_DIR.glob(f'ecfr_title{ECFR_TITLE}_text_*.ttl'))

def latest(glob_list):
    if not glob_list:
        return None
    glob_list.sort()
    return glob_list[-1]

struct_path = latest(struct_glob)
xml_path = latest(xml_glob)
ttl_path = latest(ttl_glob)
text_path = latest(text_glob)

print('Structure JSON:', struct_path)
print('Full XML:', xml_path)
print('Structure TTL:', ttl_path)
print('Text TTL:', text_path)

missing = [p for p in [struct_path, xml_path, ttl_path, text_path] if p is None]
if missing:
    print('NOT READY: missing one or more files')
else:
    print('READY: all required files present')


Structure JSON: /Users/michaelmoore/GitHub/neo4j-fibo-sec/data/sec/ecfr_title17_structure_2026-02-25.json
Full XML: /Users/michaelmoore/GitHub/neo4j-fibo-sec/data/sec/ecfr_title17_full_2026-02-25.xml
Structure TTL: /Users/michaelmoore/GitHub/neo4j-fibo-sec/data/sec/ecfr_title17_structure_2026-02-25.ttl
Text TTL: /Users/michaelmoore/GitHub/neo4j-fibo-sec/data/sec/ecfr_title17_text_2026-02-25.ttl
READY: all required files present


### 3d) Parse eCFR Structure JSON into Python Dict

Walk the eCFR hierarchy JSON (Title → Chapter → Part → Subpart → Section) and
build `sec_nodes` (uri → properties) and `broader_rels` (child, parent) pairs.
Skips auxiliary node types (`Volume`, `DescendantRange`).

In [77]:
import json
import re
import hashlib
from urllib.parse import quote

# Auxiliary types from the JSON we do not want in the graph
SKIP_TYPES = {'volume', 'descendant_range', 'descendantrange'}

sec_nodes: dict   = {}   # uri_str -> {uri, prefLabel, notation, nodeLabel}
broader_rels: list = []  # (child_uri_str, parent_uri_str)   — child broader parent

def _node_uri_str(node, parent_uri_str=None):
    node_type  = (node.get('type') or node.get('node_type') or 'node').lower()
    identifier = node.get('identifier') or node.get('node_id') or node.get('slug')
    if identifier:
        safe_id = quote(str(identifier), safe='.-_~')
        return f'https://www.ecfr.gov/cfr/{node_type}/{safe_id}'
    label = node.get('label') or node.get('heading') or node.get('title') or ''
    base  = f'{node_type}|{label}|{parent_uri_str or ""}'
    return f'https://www.ecfr.gov/cfr/{node_type}/hash/{hashlib.sha1(base.encode()).hexdigest()}'

def _neo4j_label(node_type_str):
    """e.g. 'subject-group' → 'Subject_Group'"""
    return re.sub(r'[^a-zA-Z0-9_]', '_', node_type_str.title())

def walk_json(node, parent_uri_str=None):
    node_type = (node.get('type') or node.get('node_type') or 'node').lower()
    if node_type in SKIP_TYPES:
        return

    uri_str  = _node_uri_str(node, parent_uri_str)
    label    = _neo4j_label(node_type)
    pref     = (node.get('label') or node.get('heading') or
                node.get('title') or node.get('name') or '').strip()
    notation = str(node.get('identifier') or node.get('number') or
                   node.get('section') or node.get('part') or
                   node.get('chapter') or '').strip()

    if uri_str not in sec_nodes:
        sec_nodes[uri_str] = {
            'uri':       uri_str,
            'prefLabel': pref,
            'notation':  notation,
            'nodeLabel': label,
        }

    if parent_uri_str:
        broader_rels.append((uri_str, parent_uri_str))   # child broader parent

    for child in (node.get('children') or []):
        walk_json(child, uri_str)

# ── load ──────────────────────────────────────────────────────────────────────
structure_files = sorted(SEC_DATA_DIR.glob(f'ecfr_title{ECFR_TITLE}_structure_*.json'))
if not structure_files:
    raise FileNotFoundError('Structure JSON not found — run Download Data first')
structure_path = structure_files[-1]
print(f'Parsing: {structure_path.name}')

sec_nodes.clear()
broader_rels.clear()
walk_json(json.loads(structure_path.read_text()))

type_counts: dict = {}
for n in sec_nodes.values():
    type_counts[n['nodeLabel']] = type_counts.get(n['nodeLabel'], 0) + 1

print(f'Total nodes: {len(sec_nodes):,}')
for lbl, cnt in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f'  {lbl:<20} {cnt:>6,}')
print(f'BROADER rels: {len(broader_rels):,}')


Parsing: ecfr_title17_structure_2026-02-25.json
Total nodes: 3,943
  Section               3,526
  Subject_Group           146
  Part                    129
  Appendix                 85
  Subpart                  43
  Hed1                      8
  Chapter                   3
  Subchapter                2
  Title                     1
BROADER rels: 4,117


### 3e) Parse Text TTL into sec_texts Dict

Load the text TTL generated in step 3a and build `sec_texts`
(`{sec_uri → text_str}`) for attaching to Section nodes.

In [78]:
from rdflib import Graph, Namespace

ECFR = Namespace('https://www.ecfr.gov/cfr/')

text_files = sorted(SEC_DATA_DIR.glob(f'ecfr_title{ECFR_TITLE}_text_*.ttl'))
if not text_files:
    raise FileNotFoundError('Text TTL not found — run the Extract Section Text cell first')
text_path = text_files[-1]
print(f'Parsing: {text_path.name}')

tg = Graph()
tg.parse(str(text_path), format='turtle')

sec_texts: dict = {}   # sec_uri_str -> text_str
for sec_uri, _, text_uri in tg.triples((None, ECFR.hasText, None)):
    text_lit = tg.value(text_uri, ECFR.text)
    if text_lit:
        sec_texts[str(sec_uri)] = str(text_lit)

print(f'Sections with text: {len(sec_texts):,}')


Parsing: ecfr_title17_text_2026-02-25.ttl
Sections with text: 3,525


### 3f) Build PG Graph and Load into Neo4j (IGNORE strategy)

Assemble a clean property graph from `sec_nodes`, `broader_rels`, and
`sec_texts`, then load into Neo4j via rdflib-neo4j using the IGNORE strategy —
the same approach as FIBO in Section 2.

Node labels (`Section`, `Chapter`, etc.) and relationship types (`BROADER`,
`HAS_TEXT`) come from a custom `SEC_PG` namespace; IGNORE strips the namespace
so only the local name lands in Neo4j.

In [79]:
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import RDF
from rdflib_neo4j import Neo4jStoreConfig, Neo4jStore, HANDLE_VOCAB_URI_STRATEGY

SEC_PG = Namespace('https://ecfr.neo4j/pg#')

# Discover all node labels from the JSON walk
node_labels = {n['nodeLabel'] for n in sec_nodes.values()}
PG_TYPES    = {lbl: SEC_PG[lbl] for lbl in node_labels}

pg = Graph()

# Structural nodes
for n in sec_nodes.values():
    subj = URIRef(n['uri'])
    pg.add((subj, RDF.type, PG_TYPES[n['nodeLabel']]))
    if n['prefLabel']:
        pg.add((subj, SEC_PG.prefLabel, Literal(n['prefLabel'])))
    if n['notation']:
        pg.add((subj, SEC_PG.notation,  Literal(n['notation'])))

# BROADER relationships  (child -[BROADER]-> parent)
for child_uri_str, parent_uri_str in broader_rels:
    pg.add((URIRef(child_uri_str), SEC_PG.BROADER, URIRef(parent_uri_str)))

# Text nodes + HAS_TEXT relationships
for sec_uri_str, text in sec_texts.items():
    sec_uri  = URIRef(sec_uri_str)
    text_uri = URIRef(sec_uri_str + '/text')
    pg.add((text_uri, RDF.type,        SEC_PG.Text))
    pg.add((text_uri, SEC_PG.text,     Literal(text)))
    pg.add((sec_uri,  SEC_PG.HAS_TEXT, text_uri))

print(f'PG graph: {len(pg):,} triples')
print(f'  Structural nodes : {len(sec_nodes):,}')
print(f'  BROADER rels     : {len(broader_rels):,}')
print(f'  Text nodes       : {len(sec_texts):,}')

# ── Load into Neo4j ───────────────────────────────────────────────────────────
config = Neo4jStoreConfig(
    auth_data={
        'uri':      NEO4J_URI,
        'user':     NEO4J_USER,
        'pwd':      NEO4J_PWD,
        'database': NEO4J_DB,
    },
    handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.IGNORE,
    batching=True,
    batch_size=5000,
)

store = Neo4jStore(config=config)
store.open(configuration=None, create=True)
print('\nLoading into Neo4j...')
for triple in pg:
    store.add(triple)
store.commit()
store.close(True)
print('Done.')


PG graph: 26,513 triples
  Structural nodes : 3,943
  BROADER rels     : 4,117
  Text nodes       : 3,525
Uniqueness constraint on :Resource(uri) found. 
                
                
The store is now: Open
Uniqueness constraint on :Resource(uri) found. 
                
                
The store is now: Open

Loading into Neo4j...
The store is now: Closed
IMPORTED 26513 TRIPLES
Done.


### 3g) Verify Node and Relationship Counts

In [80]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PWD))
with driver.session(database=NEO4J_DB) as session:
    nodes = session.run('MATCH (n) RETURN count(n) AS c').single()['c']
    rels  = session.run('MATCH ()-[r]->() RETURN count(r) AS c').single()['c']

    labels_raw = session.run(
        'MATCH (n) UNWIND labels(n) AS lbl '
        'RETURN lbl, count(*) AS cnt ORDER BY cnt DESC LIMIT 25'
    ).data()

    rel_raw = session.run(
        'MATCH ()-[r]->() RETURN type(r) AS rtype, count(r) AS cnt '
        'ORDER BY cnt DESC LIMIT 20'
    ).data()

driver.close()

print(f'Total nodes         : {nodes:,}')
print(f'Total relationships : {rels:,}')
print()
print('Node labels:')
for row in labels_raw:
    print(f'  {row["lbl"]:<40} {row["cnt"]:>8,}')
print()
print('Relationship types:')
for row in rel_raw:
    print(f'  {row["rtype"]:<40} {row["cnt"]:>8,}')


Total nodes         : 22,060
Total relationships : 31,587

Node labels:
  Resource                                   22,060
  Individual                                 11,627
  Section                                     3,526
  Text                                        3,525
  Class                                       2,941
  Subject_Group                                 146
  Part                                          129
  Appendix                                       85
  Subpart                                        43
  Hed1                                            8
  Chapter                                         3
  ConceptScheme                                   2
  Subchapter                                      2
  Title                                           1

Relationship types:
  BROADER                                     4,117
  SUBCLASS_OF                                 3,717
  HAS_TEXT                                    3,525
  IDENTIFIES           

### 3h) Parse Paragraph Structure (OpenAI)

For each `Text` node, send the section text to **OpenAI** (`gpt-4o-mini`) to
extract the hierarchical paragraph structure. CFR sections are highly variable —
LLM parsing handles all enumeration formats reliably.

Relationships produced:
- `(:Text)-[:HAS_PARA]->(:Paragraph)` — top-level paragraphs
- `(:Paragraph)-[:HAS_PARA]->(:Paragraph)` — structural outline nesting
- `(:Paragraph)-[:NEXT]->(:Paragraph)` — literal reading-order sequence

Results cached to `ecfr_title17_paragraphs.jsonl` — run is fully resumable.

In [3]:
import json
import re
import time
from pathlib import Path
from tqdm import tqdm
# Uses: client, CHAT_MODEL_NAME, CHAT_MODEL_TEMPERATURE, CHAT_MODEL_MAX_TOKENS
# defined in the #genai credentials cell above.

# ── Prompt ────────────────────────────────────────────────────────────────────
PARSE_PROMPT = """\
You are parsing a section from the U.S. Code of Federal Regulations (CFR).

Extract the complete paragraph hierarchy from the text below.

CFR sections use a variety of enumeration formats — often mixed within the same section:

PARENTHESIZED (most common):
  (a), (b), (c) ...          lowercase letters    → typically Level 1
  (1), (2), (3) ...          arabic numbers       → typically Level 2
  (i), (ii), (iii) ...       roman numerals       → typically Level 3
  (A), (B), (C) ...          uppercase letters    → typically Level 4

DOTTED / PERIOD-TERMINATED:
  1., 2., 3. ...             arabic with period
  a., b., c. ...             lowercase with period
  I., II., III. ...          Roman uppercase with period

ITEM / INSTRUCTION labels:
  "Item 1.", "Item 2." ...   appendix-style items
  "Instruction to paragraph (x):"  — instruction block following its target

DEFINITION-STYLE (no marker):
  Term. Definition text.     a term (often italicised) followed by its definition

TABLES and SCHEDULES:
  Treat each logical row/entry as a child paragraph with its label as the marker.

GENERAL RULES:
- Preserve markers EXACTLY as they appear: "(a)", "1.", "Item 3", "(iii)"
- Any introductory / body text before the first marker → marker: ""
- A paragraph's "text" is its OWN prose only — not the text of its children
- If a paragraph ends with a colon or dash and introduces a list, the list items
  are its "children"
- Instruction blocks go as a child of their target paragraph
- Ignore trailing citation lines and authority lines at the end of the section

Return a JSON object with key "paragraphs" containing the array:
{{
  "paragraphs": [
    {{
      "marker":   "(a)",
      "text":     "complete text of this item only, including the marker",
      "children": []
    }}
  ]
}}

Section text to parse:
---
{text}
---"""


MAX_TEXT_CHARS = 12_000   # truncate very long sections to avoid hangs / huge cost

def _call_llm(text, retries=3):
    """Call gpt-4o-mini and return the parsed paragraph list."""
    # Truncate excessively long texts (e.g. appendix tables with hundreds of items)
    if len(text) > MAX_TEXT_CHARS:
        text = text[:MAX_TEXT_CHARS] + "\n[... truncated ...]"

    prompt = PARSE_PROMPT.format(text=text)
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=CHAT_MODEL_NAME,
                temperature=CHAT_MODEL_TEMPERATURE,
                max_tokens=CHAT_MODEL_MAX_TOKENS,
                response_format={"type": "json_object"},
                timeout=60,          # hard 60-second timeout per call
                messages=[{"role": "user", "content": prompt}],
            )
            result = json.loads(resp.choices[0].message.content)
            # Accept {"paragraphs": [...]} or any top-level list value
            if isinstance(result, list):
                return result
            if "paragraphs" in result:
                return result["paragraphs"]
            for v in result.values():
                if isinstance(v, list):
                    return v
            return []
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                raise


def _flatten(items, base_uri, text_uri, counter, seq,
             para_nodes, text_has_para, para_has_para,
             parent_uri=None, depth=0):
    """
    Depth-first traversal of the LLM-returned nested JSON.

    seq   — accumulates every para_uri in document (reading) order;
            caller builds NEXT from consecutive seq pairs.
    depth — nesting depth (0 = top-level under Text).
    """
    for item in (items or []):
        counter[0] += 1
        para_uri = f'{base_uri}/para/{counter[0]}'

        para_nodes[para_uri] = {
            'uri':    para_uri,
            'marker': item.get('marker') or '',
            'text':   item.get('text')   or '',
            'depth':  depth,
        }

        # HAS_PARA: structural nesting
        if parent_uri:
            para_has_para.append((parent_uri, para_uri))
        else:
            text_has_para.append((text_uri, para_uri))

        # Append to linear reading-order sequence
        seq.append(para_uri)

        # Recurse into children before advancing to next sibling
        _flatten(item.get('children', []), base_uri, text_uri, counter, seq,
                 para_nodes, text_has_para, para_has_para,
                 parent_uri=para_uri, depth=depth + 1)


def parse_texts(texts, cache_path=None):
    """
    LLM-parse the paragraph structure of every section text.

    Parameters
    ----------
    texts      : {sec_uri -> text_str}
    cache_path : Path to .jsonl file for caching / resuming

    Returns
    -------
    para_nodes    : {para_uri -> {uri, marker, text, depth}}
    text_has_para : [(text_uri,   para_uri), ...]   Text → top-level Paragraph
    para_has_para : [(parent_uri, child_uri), ...]  HAS_PARA structural nesting
    next_rels     : [(prev_uri,   next_uri),  ...]  NEXT reading-order chain
    """
    cache_path = Path(cache_path or
                      SEC_DATA_DIR / f'ecfr_title{ECFR_TITLE}_paragraphs.jsonl')

    # Load cached results
    done = {}
    if cache_path.exists():
        with open(cache_path) as f:
            for line in f:
                rec = json.loads(line)
                done[rec['sec_uri']] = rec['items']
        print(f'Cache loaded: {len(done):,} sections already parsed')

    para_nodes    = {}
    text_has_para = []
    para_has_para = []
    next_rels     = []
    errors        = []

    WORKERS = 20   # concurrent OpenAI calls; raise if rate limits allow

    remaining = {k: v for k, v in texts.items() if k not in done}
    print(f'Sections to parse : {len(remaining):,}  (skipping {len(done):,} cached)  workers={WORKERS}')

    import threading
    from concurrent.futures import ThreadPoolExecutor, as_completed

    cache_lock = threading.Lock()

    def _process(sec_uri, text):
        if not text.strip():
            return sec_uri, []
        return sec_uri, _call_llm(text)

    with open(cache_path, 'a') as cache_f:
        futures = {}
        with ThreadPoolExecutor(max_workers=WORKERS) as pool:
            for sec_uri, text in remaining.items():
                futures[pool.submit(_process, sec_uri, text)] = sec_uri

            for fut in tqdm(as_completed(futures), total=len(futures),
                            desc='Parsing', unit='sec'):
                sec_uri = futures[fut]
                try:
                    _, items = fut.result()
                    done[sec_uri] = items
                    with cache_lock:
                        cache_f.write(json.dumps({'sec_uri': sec_uri, 'items': items}) + '\n')
                        cache_f.flush()
                except Exception as e:
                    errors.append((sec_uri, str(e)))
                    done[sec_uri] = []

    # Flatten all results (cached + newly parsed)
    for sec_uri, items in done.items():
        counter  = [0]
        seq      = []
        base_uri = sec_uri
        text_uri = sec_uri + '/text'
        _flatten(items, base_uri, text_uri, counter, seq,
                 para_nodes, text_has_para, para_has_para)
        for i in range(len(seq) - 1):
            next_rels.append((seq[i], seq[i + 1]))

    if errors:
        print(f'\nFailed: {len(errors):,} sections')
        for uri, err in errors[:5]:
            print(f'  {uri}: {err}')

    return para_nodes, text_has_para, para_has_para, next_rels


# ── Read section texts from graph, then run ───────────────────────────────────
from neo4j import GraphDatabase as _GDB

_driver = _GDB.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PWD))
with _driver.session(database=NEO4J_DB) as _s:
    _rows = _s.run("""
        MATCH (sec:Section)-[:HAS_TEXT]->(t:Text)
        RETURN sec.uri AS sec_uri, t.text AS text
    """).data()
_driver.close()

graph_texts = {r['sec_uri']: r['text'] for r in _rows}
print(f'Sections with text in graph: {len(graph_texts):,}')

para_nodes, text_has_para, para_has_para, next_rels = parse_texts(graph_texts)

print(f'Paragraphs          : {len(para_nodes):,}')
print(f'Text  -[HAS_PARA]-> : {len(text_has_para):,}')
print(f'Para  -[HAS_PARA]-> : {len(para_has_para):,}')
print(f'Para  -[NEXT]->     : {len(next_rels):,}')


Sections with text in graph: 3,503
Sections to parse : 3,503  (skipping 0 cached)  workers=20


Parsing: 100%|███████████████████████████| 3503/3503 [1:00:47<00:00,  1.04s/sec]


Failed: 942 sections
  https://www.ecfr.gov/cfr/section/240.14d-100: Unterminated string starting at: line 38 column 23 (char 4309)
  https://www.ecfr.gov/cfr/section/140.735-8: Unterminated string starting at: line 67 column 19 (char 5095)
  https://www.ecfr.gov/cfr/section/240.12g-3: Expecting value: line 50 column 20 (char 4246)
  https://www.ecfr.gov/cfr/section/1.55: Unterminated string starting at: line 10 column 15 (char 1041)
  https://www.ecfr.gov/cfr/section/40.1: Unterminated string starting at: line 70 column 19 (char 4990)
Paragraphs          : 9,704
Text  -[HAS_PARA]-> : 5,087
Para  -[HAS_PARA]-> : 4,617
Para  -[NEXT]->     : 7,296


### 3i) Load Paragraph Graph into Neo4j

In [4]:
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import RDF
from rdflib_neo4j import Neo4jStoreConfig, Neo4jStore, HANDLE_VOCAB_URI_STRATEGY

SEC_PG = Namespace('https://ecfr.neo4j/pg#')

pg = Graph()

for p in para_nodes.values():
    subj = URIRef(p['uri'])
    pg.add((subj, RDF.type,       SEC_PG.Paragraph))
    if p['marker']:
        pg.add((subj, SEC_PG.marker, Literal(p['marker'])))
    if p['text']:
        pg.add((subj, SEC_PG.text,   Literal(p['text'])))
    pg.add((subj, SEC_PG.depth,  Literal(p['depth'])))

for text_uri_str, para_uri_str in text_has_para:
    pg.add((URIRef(text_uri_str), SEC_PG.HAS_PARA, URIRef(para_uri_str)))

for parent_uri_str, child_uri_str in para_has_para:
    pg.add((URIRef(parent_uri_str), SEC_PG.HAS_PARA, URIRef(child_uri_str)))

for prev_uri_str, next_uri_str in next_rels:
    pg.add((URIRef(prev_uri_str), SEC_PG.NEXT, URIRef(next_uri_str)))

print(f'PG graph: {len(pg):,} triples')

config = Neo4jStoreConfig(
    auth_data={
        'uri':      NEO4J_URI,
        'user':     NEO4J_USER,
        'pwd':      NEO4J_PWD,
        'database': NEO4J_DB,
    },
    handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.IGNORE,
    batching=True,
    batch_size=5000,
)
store = Neo4jStore(config=config)
store.open(configuration=None, create=True)
print('Loading paragraph graph into Neo4j...')
for triple in pg:
    store.add(triple)
store.commit()
store.close(True)
print('Done.')


PG graph: 54,566 triples
Uniqueness constraint on :Resource(uri) found. 
                
                
The store is now: Open
Uniqueness constraint on :Resource(uri) found. 
                
                
The store is now: Open
Loading paragraph graph into Neo4j...
The store is now: Closed
IMPORTED 54566 TRIPLES
Done.


### 3j) Text Embeddings

Embed two node types using Azure OpenAI (`text-embedding-3-large`, 3072 dims):

| Step | Nodes | Property | Index |
|------|-------|----------|-------|
| 3j-a | `Paragraph` | `textVector` | `secParagraphSearch` |
| 3j-b | `Class` | `labelVector` | `fiboClassSearch` |

Run **3j-a** then **3j-b** independently — both are idempotent (skip already-embedded nodes).

In [ ]:
# 3j-a) Embed Paragraph.text → textVector

from openai import AzureOpenAI
from neo4j import GraphDatabase
import time
from datetime import timedelta

_azure = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
)

PARA_BATCH_SIZE   = 100    # texts per Azure embedding call  (max 2048)
PARA_FETCH_SIZE   = 2000   # nodes fetched from Neo4j per round
PARA_MAX_CHARS    = 8000   # skip paragraphs longer than this

def _embed_batch(texts):
    resp = _azure.embeddings.create(
        model=EMBEDDING_DEPLOYMENT_NAME,
        input=texts,
        dimensions=EMBEDDING_DIMENSIONS,
    )
    return [x.embedding for x in sorted(resp.data, key=lambda x: x.index)]

def embed_paragraphs():
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PWD))
    t0 = time.time()
    done = skipped = 0

    with driver.session(database=NEO4J_DB) as s:
        total = s.run("""
            MATCH (n:Paragraph)
            WHERE n.text IS NOT NULL AND n.textVector IS NULL
            RETURN count(n) AS c
        """).single()['c']
    print(f"Paragraphs to embed: {total:,}")

    while True:
        with driver.session(database=NEO4J_DB) as s:
            rows = s.run("""
                MATCH (n:Paragraph)
                WHERE n.text IS NOT NULL AND n.textVector IS NULL
                RETURN elementId(n) AS id, n.text AS text
                LIMIT $lim
            """, lim=PARA_FETCH_SIZE).data()

        if not rows:
            break

        valid = [r for r in rows if len(r['text']) <= PARA_MAX_CHARS]
        skipped += len(rows) - len(valid)

        with driver.session(database=NEO4J_DB) as s:
            for i in range(0, len(valid), PARA_BATCH_SIZE):
                batch = valid[i:i + PARA_BATCH_SIZE]
                vecs  = _embed_batch([r['text'] for r in batch])
                s.run("""
                    UNWIND $rows AS row
                    MATCH (n:Paragraph) WHERE elementId(n) = row.id
                    CALL db.create.setNodeVectorProperty(n, 'textVector', row.vec)
                """, rows=[{'id': r['id'], 'vec': v} for r, v in zip(batch, vecs)])
                done += len(batch)
                elapsed = time.time() - t0
                rate    = done / elapsed if elapsed > 0 else 0
                eta     = timedelta(seconds=int((total - done) / rate)) if rate > 0 else "?"
                print(f"\r  {done:,}/{total:,}  {rate:.1f}/s  ETA {eta}  skipped {skipped}",
                      end="", flush=True)

    driver.close()
    print(f"\nDone — {done:,} embedded in {timedelta(seconds=int(time.time()-t0))}")

embed_paragraphs()


In [ ]:
# Create vector indexes (run once — IF NOT EXISTS makes it safe to re-run)
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PWD))
with driver.session(database=NEO4J_DB) as s:
    s.run("""
        CREATE VECTOR INDEX secParagraphSearch IF NOT EXISTS
        FOR (n:Paragraph) ON (n.textVector)
        OPTIONS {indexConfig: {
            `vector.dimensions`: 3072,
            `vector.similarity_function`: 'cosine'
        }}
    """)
    print("secParagraphSearch index: OK")

    s.run("""
        CREATE VECTOR INDEX fiboClassSearch IF NOT EXISTS
        FOR (n:Class) ON (n.labelVector)
        OPTIONS {indexConfig: {
            `vector.dimensions`: 3072,
            `vector.similarity_function`: 'cosine'
        }}
    """)
    print("fiboClassSearch index: OK")

driver.close()


In [ ]:
# 3j-b) Embed Class.prefLabel → labelVector

from openai import AzureOpenAI
from neo4j import GraphDatabase
import time
from datetime import timedelta

_azure = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
)

CLASS_BATCH_SIZE = 100
CLASS_FETCH_SIZE = 2000

def _embed_batch_labels(texts):
    resp = _azure.embeddings.create(
        model=EMBEDDING_DEPLOYMENT_NAME,
        input=texts,
        dimensions=EMBEDDING_DIMENSIONS,
    )
    return [x.embedding for x in sorted(resp.data, key=lambda x: x.index)]

def embed_fibo_labels():
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PWD))
    t0 = time.time()
    done = 0

    with driver.session(database=NEO4J_DB) as s:
        total = s.run("""
            MATCH (n:Class)
            WHERE n.prefLabel IS NOT NULL AND n.labelVector IS NULL
            RETURN count(n) AS c
        """).single()['c']
    print(f"Class nodes to embed: {total:,}")

    while True:
        with driver.session(database=NEO4J_DB) as s:
            rows = s.run("""
                MATCH (n:Class)
                WHERE n.prefLabel IS NOT NULL AND n.labelVector IS NULL
                RETURN elementId(n) AS id, n.prefLabel AS label
                LIMIT $lim
            """, lim=CLASS_FETCH_SIZE).data()

        if not rows:
            break

        with driver.session(database=NEO4J_DB) as s:
            for i in range(0, len(rows), CLASS_BATCH_SIZE):
                batch = rows[i:i + CLASS_BATCH_SIZE]
                vecs  = _embed_batch_labels([r['label'] for r in batch])
                s.run("""
                    UNWIND $rows AS row
                    MATCH (n:Class) WHERE elementId(n) = row.id
                    CALL db.create.setNodeVectorProperty(n, 'labelVector', row.vec)
                """, rows=[{'id': r['id'], 'vec': v} for r, v in zip(batch, vecs)])
                done += len(batch)
                elapsed = time.time() - t0
                rate    = done / elapsed if elapsed > 0 else 0
                eta     = timedelta(seconds=int((total - done) / rate)) if rate > 0 else "?"
                print(f"\r  {done:,}/{total:,}  {rate:.1f}/s  ETA {eta}",
                      end="", flush=True)

    driver.close()
    print(f"\nDone — {done:,} embedded in {timedelta(seconds=int(time.time()-t0))}")

embed_fibo_labels()


## 4) Infer GOVERNS Relationships

Link FIBO concepts to CFR paragraphs using **vector cosine similarity**.
For each `FIBO:Class` node that has a `labelVector`, the `secParagraphSearch`
ANN index finds the most semantically similar `Paragraph` nodes.

Results: `(para:Paragraph)-[:GOVERNS {similarity_score, method}]->(fibo:Class)`

Indexes required (created in §3j):
- `fiboLabelSearch` — `Class.labelVector`
- `secParagraphSearch` — `Paragraph.textVector`
- `secTextSearch` — `SEC.textVector`


### 4a) Clear Existing GOVERNS Relationships

In [14]:
# Clear existing GOVERNS relationships before re-running linkage
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PWD))
with driver.session(database=NEO4J_DB) as session:
    result = session.run('MATCH ()-[r:GOVERNS]->() DELETE r RETURN count(r) AS deleted')
    print(f'Deleted {result.single()["deleted"]:,} existing GOVERNS relationships')
driver.close()


Deleted 602 existing GOVERNS relationships


### 4b) Vector Similarity Inference

For each `FIBO:Class`, query the `secParagraphSearch` ANN index to find
the top-K CFR `Paragraph` nodes by cosine similarity.
Write `(para)-[:GOVERNS]->(fibo)` for any pair whose score exceeds `MIN_SCORE`.

**Parameters**
| Name | Default | Meaning |
|------|---------|------|
| `TOP_K` | 10 | Candidate paragraphs per FIBO class |
| `MIN_SCORE` | 0.60 | Minimum cosine similarity to create a link |


In [15]:
# 4b) Vector GOVERNS — write relationships
from neo4j import GraphDatabase

TOP_K     = 350   # ANN search width — threshold does the real filtering
MIN_SCORE = 0.75  # cosine similarity threshold (0-1); raise to tighten, lower to broaden

GOVERNS_WRITE = """
MATCH (f:FIBO:Class)
WHERE f.labelVector IS NOT NULL
CALL(f) {
    CALL db.index.vector.queryNodes("secTextSearch", $top_k, f.labelVector)
    YIELD node AS sec_node, score
    WHERE score > $min_score
    MERGE (sec_node)-[g:GOVERNS]->(f)
    ON CREATE SET g.similarity_score = score,
                  g.method           = 'vector'
    ON MATCH  SET g.similarity_score = score
    RETURN count(*) AS c
}
RETURN sum(c) AS total_written
"""

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PWD))
with driver.session(database=NEO4J_DB) as session:
    result = session.run(GOVERNS_WRITE, top_k=TOP_K, min_score=MIN_SCORE)
    total = result.single()["total_written"]
    print(f"GOVERNS written : {total:,}")
driver.close()


GOVERNS written : 8,702


### 4c) Verify GOVERNS

In [22]:
# 4c) Verify vector GOVERNS results
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PWD))
with driver.session(database=NEO4J_DB) as session:

    # Total GOVERNS relationships
    total = session.run(
        "MATCH ()-[r:GOVERNS]->() RETURN count(r) AS c"
    ).single()["c"]

    # Total :SEC nodes (Text + Paragraph combined index)
    total_sec = session.run(
        "MATCH (n:SEC) RETURN count(n) AS c"
    ).single()["c"]

    # Distinct :SEC nodes linked via GOVERNS
    sec_linked = session.run(
        "MATCH (n:SEC)-[:GOVERNS]->() RETURN count(DISTINCT n) AS c"
    ).single()["c"]

    # Breakdown by node type among linked nodes
    breakdown = session.run("""
        MATCH (n:SEC)-[:GOVERNS]->()
        WITH DISTINCT n
        RETURN
            CASE WHEN n:Paragraph THEN 'Paragraph'
                 WHEN n:Text      THEN 'Text'
                 ELSE 'Other' END AS node_type,
            count(*) AS cnt
        ORDER BY cnt DESC
    """).data()

    # Distinct FIBO classes linked
    fibo_linked = session.run(
        "MATCH ()-[:GOVERNS]->(f:FIBO:Class) RETURN count(DISTINCT f) AS c"
    ).single()["c"]

    # Total FIBO classes with labelVector
    total_fibo = session.run(
        "MATCH (f:FIBO:Class) WHERE f.labelVector IS NOT NULL RETURN count(f) AS c"
    ).single()["c"]

    # Score distribution (0.05-width buckets for finer view at high end)
    dist = session.run("""
        MATCH ()-[r:GOVERNS]->()
        RETURN
            round(r.similarity_score * 20) / 20 AS bucket,
            count(r) AS cnt
        ORDER BY bucket DESC
    """).data()

    # Top 10 FIBO classes by incoming GOVERNS count
    top_fibo = session.run("""
        MATCH (n:SEC)-[r:GOVERNS]->(f:FIBO:Class)
        WITH f, count(r) AS links, avg(r.similarity_score) AS avg_score
        RETURN f.prefLabel AS concept, links, round(avg_score*100)/100 AS avg_score
        ORDER BY links DESC LIMIT 10
    """).data()

sec_pct  = sec_linked  / total_sec  * 100 if total_sec  else 0
fibo_pct = fibo_linked / total_fibo * 100 if total_fibo else 0

print(f"GOVERNS relationships : {total:,}")
print(f"SEC nodes linked      : {sec_linked:,} / {total_sec:,}  ({sec_pct:.1f}%)")
for row in breakdown:
    print(f"  {row['node_type']:<12} {row['cnt']:,}")
print(f"FIBO classes linked   : {fibo_linked:,} / {total_fibo:,} with embeddings  ({fibo_pct:.1f}%)")
print()
print("Score distribution (0.05 buckets):")
for row in dist:
    print(f"  [{row['bucket']:.2f}]  {row['cnt']:,}")
print()
print("Top FIBO classes by incoming GOVERNS:")
for row in top_fibo:
    print(f"  {row['concept']:<50} links={row['links']:4d}  avg_score={row['avg_score']}")

driver.close()


GOVERNS relationships : 10,567
SEC nodes linked      : 3,617 / 17,194  (21.0%)
  Paragraph    2,401
  Other        788
  Text         428
FIBO classes linked   : 1,345 / 2,941 with embeddings  (45.7%)

Score distribution (0.05 buckets):
  [0.95]  8
  [0.90]  21
  [0.85]  213
  [0.80]  2,550
  [0.75]  7,775

Top FIBO classes by incoming GOVERNS:
  major swap participant                             links= 315  avg_score=0.77
  derivatives clearing organization                  links= 239  avg_score=0.79
  designated contract market                         links= 187  avg_score=0.78
  securities trades reporting                        links= 167  avg_score=0.76
  futures commission merchant                        links= 163  avg_score=0.78
  exempt issuer                                      links= 153  avg_score=0.77
  exempt security                                    links= 131  avg_score=0.77
  offering statement                                 links= 125  avg_score=0.77
  final prosp

### 4d) Section-Level Rollup (Optional)

Roll paragraph-level GOVERNS up to the containing `Section` node.
Aggregates the max similarity score per (section, FIBO-class) pair.

Result: `(sec:Section)-[:GOVERNS {similarity_score, method:'vector_rollup'}]->(fibo:Class)`


In [21]:
# 4d) Roll up SEC-node GOVERNS to Section level
# Collects signal from both Text nodes (1 hop) and Paragraph nodes (2 hops)
from neo4j import GraphDatabase

ROLLUP_QUERY = """
CALL() {
    MATCH (sec:Section)-[:HAS_TEXT]->(t:Text)-[r:GOVERNS]->(f:FIBO:Class)
    RETURN sec, f, r.similarity_score AS score
    UNION
    MATCH (sec:Section)-[:HAS_TEXT]->(:Text)-[:HAS_PARA]->(p:Paragraph)-[r:GOVERNS]->(f:FIBO:Class)
    RETURN sec, f, r.similarity_score AS score
}
WITH sec, f, max(score) AS top_score
MERGE (sec)-[g:GOVERNS]->(f)
ON CREATE SET g.similarity_score = top_score,
              g.method           = 'vector_rollup'
ON MATCH  SET g.similarity_score = top_score
RETURN count(*) AS total_written
"""

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PWD))
with driver.session(database=NEO4J_DB) as session:
    result = session.run(ROLLUP_QUERY)
    total = result.single()["total_written"]
    print(f"Section-level GOVERNS written: {total:,}")

    rows = session.run("""
        MATCH (sec:Section)-[r:GOVERNS {method:'vector_rollup'}]->(f:FIBO:Class)
        RETURN count(r) AS links,
               count(DISTINCT sec) AS sections,
               count(DISTINCT f)   AS fibo_classes
    """).single()
    print(f"Sections linked : {rows['sections']:,}")
    print(f"FIBO classes    : {rows['fibo_classes']:,}")

driver.close()


Section-level GOVERNS written: 1,865
Sections linked : 788
FIBO classes    : 482


### 4e) Sankey: Top FIBO Classes → CFR Parts

Visualises the section-level GOVERNS rollup as a Plotly Sankey.
Left nodes = FIBO classes (top N by link count), right nodes = CFR Parts.
Flow width = number of sections linked.


In [ ]:
# 4e) Sankey diagram — top FIBO classes → CFR Parts
import plotly.graph_objects as go
from neo4j import GraphDatabase

TOP_N = 20   # top FIBO classes to display

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PWD))
with driver.session(database=NEO4J_DB) as session:

    # Top N FIBO classes by rollup GOVERNS count
    top_fibo = session.run("""
        MATCH (:Section)-[r:GOVERNS {method:'vector_rollup'}]->(f:FIBO:Class)
        RETURN f.prefLabel AS label, count(r) AS cnt
        ORDER BY cnt DESC LIMIT $n
    """, n=TOP_N).data()

    labels = [r['label'] for r in top_fibo]

    # Part-level aggregation for those classes
    link_rows = session.run("""
        MATCH (sec:Section)-[r:GOVERNS {method:'vector_rollup'}]->(f:FIBO:Class)
        WHERE f.prefLabel IN $labels
        MATCH (sec)-[:BROADER]->(part:Part)
        RETURN f.prefLabel                              AS fibo_label,
               coalesce(part.notation, part.prefLabel)  AS part_label,
               count(r)                                 AS cnt
        ORDER BY cnt DESC
    """, labels=labels).data()

driver.close()

# Build node lists
fibo_nodes = list(dict.fromkeys(r['fibo_label'] for r in link_rows))  # preserve order
part_nodes  = list(dict.fromkeys(r['part_label']  for r in link_rows))
all_nodes   = fibo_nodes + part_nodes
node_idx    = {n: i for i, n in enumerate(all_nodes)}

fig = go.Figure(go.Sankey(
    arrangement='snap',
    node=dict(
        label     = all_nodes,
        pad       = 15,
        thickness = 20,
        color     = ['#4C78A8'] * len(fibo_nodes) + ['#F58518'] * len(part_nodes),
    ),
    link=dict(
        source = [node_idx[r['fibo_label']] for r in link_rows],
        target = [node_idx[r['part_label']]  for r in link_rows],
        value  = [r['cnt']                   for r in link_rows],
    ),
))

fig.update_layout(
    title_text=f"Top {TOP_N} FIBO Classes → CFR Parts  (vector_rollup GOVERNS)",
    font_size=11,
    height=700,
)
fig.show()


## 5) Patch FIBO SUBCLASS_OF Hierarchy

Some FIBO nodes loaded with the IGNORE strategy have no outgoing `SUBCLASS_OF`
relationship — their position in the ontology hierarchy is implicit in their URI.

This cell reconstructs that hierarchy by parsing each URI into path segments,
MERGEing intermediate `Class` nodes, and chaining them with `SUBCLASS_OF`.

**URI pattern:** `https://spec.edmcouncil.org/fibo/ontology/{Domain}/{Module}/{Ontology}/{Concept}`

Result: `(n)-[:SUBCLASS_OF]->(Ontology)-[:SUBCLASS_OF]->(Module)-[:SUBCLASS_OF]->(Domain)-[:SUBCLASS_OF]->(base)`


In [23]:
# 5) Patch FIBO SUBCLASS_OF hierarchy from URI path segments
from neo4j import GraphDatabase
from tqdm import tqdm

# Known FIBO URI bases — path segments after these form the hierarchy
FIBO_BASES = [
    'https://spec.edmcouncil.org/fibo/ontology',
    'https://www.omg.org/spec',
]

def uri_chain(uri):
    """
    Parse a FIBO class URI into a chain of (uri, prefLabel) pairs
    from the base root down to the immediate parent of the leaf.
    The leaf itself (the existing node n) is excluded.

    Example:
      .../fibo/ontology/FBC/ProductsAndServices/ClientsAndAccounts/PaymentDueDate
    Returns:
      [('https://.../fibo/ontology',                                   'ontology'),
       ('https://.../fibo/ontology/FBC',                                'FBC'),
       ('https://.../fibo/ontology/FBC/ProductsAndServices',            'ProductsAndServices'),
       ('https://.../fibo/ontology/FBC/ProductsAndServices/ClientsAndAccounts', 'ClientsAndAccounts')]
    """
    for base in FIBO_BASES:
        if uri.startswith(base + '/') or uri == base:
            tail     = uri[len(base):].lstrip('/')
            if not tail:
                return []                     # uri IS the base — nothing to chain
            segments = tail.split('/')
            # Root entry: the base URI itself
            chain = [(base, base.rstrip('/').split('/')[-1])]
            # Intermediate entries: all segments except the final leaf
            for i, seg in enumerate(segments[:-1]):
                chain.append((
                    base + '/' + '/'.join(segments[:i+1]),
                    seg
                ))
            return chain
    return []   # unrecognised base — skip


driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PWD))
with driver.session(database=NEO4J_DB) as session:

    # ── 1. Find orphans ────────────────────────────────────────────────────────
    orphans = session.run(
        "MATCH (n:FIBO) WHERE NOT (n)-[:SUBCLASS_OF]->() RETURN n.uri AS uri"
    ).data()
    print(f"FIBO nodes without SUBCLASS_OF : {len(orphans)}")

    # ── 2. Parse all URIs ─────────────────────────────────────────────────────
    intermediates = {}   # uri -> prefLabel  (deduplicated)
    rel_pairs     = set()  # (child_uri, parent_uri)

    for row in orphans:
        uri = row['uri']
        if not uri:
            continue
        chain = uri_chain(uri)
        if not chain:
            continue

        # Collect intermediate nodes
        for node_uri, label in chain:
            intermediates[node_uri] = label

        # Chain: each consecutive pair in chain is a SUBCLASS_OF rel
        for i in range(1, len(chain)):
            rel_pairs.add((chain[i][0], chain[i - 1][0]))

        # Leaf node n -> its immediate parent (last entry in chain)
        rel_pairs.add((uri, chain[-1][0]))

    print(f"Unique intermediate nodes      : {len(intermediates)}")
    print(f"SUBCLASS_OF pairs to merge     : {len(rel_pairs)}")

    # ── 3. MERGE intermediate Class nodes (batch) ─────────────────────────────
    node_batch = [{'uri': u, 'label': l} for u, l in intermediates.items()]
    result = session.run(
        """
        UNWIND $nodes AS row
        MERGE (m:Class {uri: row.uri})
        ON CREATE SET m.prefLabel = row.label
        SET m:FIBO
        RETURN count(*) AS c
        """,
        nodes=node_batch,
    )
    print(f"Intermediate nodes processed   : {result.single()['c']:,}")

    # ── 4. MERGE SUBCLASS_OF relationships (batch) ────────────────────────────
    rel_batch = [{'child': c, 'parent': p} for c, p in rel_pairs]
    result = session.run(
        """
        UNWIND $rels AS row
        MATCH (child:Class  {uri: row.child})
        MATCH (parent:Class {uri: row.parent})
        MERGE (child)-[:SUBCLASS_OF]->(parent)
        RETURN count(*) AS c
        """,
        rels=rel_batch,
    )
    print(f"SUBCLASS_OF rels merged        : {result.single()['c']:,}")

    # ── 5. Verify ─────────────────────────────────────────────────────────────
    remaining = session.run(
        "MATCH (n:FIBO) WHERE NOT (n)-[:SUBCLASS_OF]->() RETURN count(n) AS c"
    ).single()["c"]
    print(f"FIBO nodes still without SUBCLASS_OF : {remaining}")

driver.close()


FIBO nodes without SUBCLASS_OF : 447
Unique intermediate nodes      : 168
SUBCLASS_OF pairs to merge     : 612
Intermediate nodes processed   : 168
SUBCLASS_OF rels merged        : 612
FIBO nodes still without SUBCLASS_OF : 3
